# NVIDIA NeMo Text Processing

This notebook demonstrates two core operations from [NVIDIA NeMo Text Processing](https://github.com/NVIDIA/NeMo-text-processing):

| Operation | Direction | Use case |
|-----------|-----------|----------|
| **Text Normalization (TN)** | Written → Spoken | Prepare text before Text-to-Speech (TTS) |
| **Inverse Text Normalization (ITN)** | Spoken → Written | Clean up text after Automatic Speech Recognition (ASR) |

**Supported languages:** English, German, Spanish, French, Hungarian, Swedish, Mandarin, Arabic, Italian, Armenian, Japanese, Hindi, Korean, Vietnamese, Portuguese

**Hardware:** CPU only — no GPU required

---
Run each cell in order. The first section verifies the installation.

## 1. Verify Installation

In [ ]:
import importlib

if importlib.util.find_spec("nemo_text_processing") is None:
    print("Installing nemo_text_processing — this takes 3-5 minutes...")
    import subprocess
    subprocess.run(["pip", "install", "--quiet", "nemo_text_processing"], check=True)
    print("Done.")
else:
    print("nemo_text_processing is installed — ready to go.")

---
## 2. Text Normalization (TN) — Written → Spoken

Text Normalization converts written symbols into how they would be read aloud.  
This is the preprocessing step required before feeding text into a TTS model.

In [ ]:
from nemo_text_processing.text_normalization.normalize import Normalizer

# Create an English normalizer
normalizer = Normalizer(input_case='cased', lang='en')
print("English normalizer ready.")

In [ ]:
# Numbers
examples = [
    "12 kg",
    "The temperature is -5°C",
    "$4.99",
    "1,000,000 people",
    "Chapter 7",
]

print("=== Numbers ===")
for text in examples:
    result = normalizer.normalize(text, verbose=False)
    print(f"  {text!r:35} → {result!r}")

In [ ]:
# Dates and times
examples = [
    "The meeting is on 05/12/2024",
    "See you at 3:30pm",
    "The event starts at 9:00 AM",
    "Born on January 1st, 1990",
]

print("=== Dates and Times ===")
for text in examples:
    result = normalizer.normalize(text, verbose=False)
    print(f"  {text!r:45} → {result!r}")

In [ ]:
# Abbreviations, measurements and symbols
examples = [
    "Dr. Smith will see you now",
    "The speed limit is 60 mph",
    "Mix 2 tbsp of sugar",
    "The file is 3.5 GB",
    "She scored 95% on the exam",
]

print("=== Abbreviations and Measurements ===")
for text in examples:
    result = normalizer.normalize(text, verbose=False)
    print(f"  {text!r:40} → {result!r}")

### 2.1 Multilingual Text Normalization

Swap the `lang` parameter to normalize text in any of the 15 supported languages.

In [ ]:
# German
normalizer_de = Normalizer(input_case='cased', lang='de')

examples_de = [
    "Die Temperatur beträgt -3°C",
    "Das kostet 12,50 €",
    "Kapitel 5",
]

print("=== German (de) ===")
for text in examples_de:
    result = normalizer_de.normalize(text, verbose=False)
    print(f"  {text!r:35} → {result!r}")

In [ ]:
# Spanish
normalizer_es = Normalizer(input_case='cased', lang='es')

examples_es = [
    "El precio es $25.99",
    "Tiene 3 hijos",
    "La reunión es el 15/06/2024",
]

print("=== Spanish (es) ===")
for text in examples_es:
    result = normalizer_es.normalize(text, verbose=False)
    print(f"  {text!r:35} → {result!r}")

---
## 3. Inverse Text Normalization (ITN) — Spoken → Written

Inverse Text Normalization converts spoken-form text back into written conventions.  
This is the post-processing step applied to raw ASR output to make it readable.

In [ ]:
from nemo_text_processing.inverse_text_normalization.inverse_normalize import InverseNormalizer

inverse_normalizer = InverseNormalizer(lang='en')
print("English inverse normalizer ready.")

In [ ]:
# Numbers and money
examples = [
    "twelve kilograms",
    "four dollars and ninety nine cents",
    "one million people",
    "negative five degrees",
    "ninety five percent",
]

print("=== Numbers and Money (ITN) ===")
for text in examples:
    result = inverse_normalizer.inverse_normalize(text, verbose=False)
    print(f"  {text!r:45} → {result!r}")

In [ ]:
# Dates and times
examples = [
    "january first two thousand and twenty four",
    "three thirty p m",
    "nine o'clock in the morning",
]

print("=== Dates and Times (ITN) ===")
for text in examples:
    result = inverse_normalizer.inverse_normalize(text, verbose=False)
    print(f"  {text!r:50} → {result!r}")

In [ ]:
# Measurements
examples = [
    "sixty miles per hour",
    "three point five gigabytes",
    "two tablespoons of sugar",
]

print("=== Measurements (ITN) ===")
for text in examples:
    result = inverse_normalizer.inverse_normalize(text, verbose=False)
    print(f"  {text!r:40} → {result!r}")

### 3.1 Multilingual Inverse Text Normalization

In [ ]:
# German ITN
inverse_normalizer_de = InverseNormalizer(lang='de')

examples_de = [
    "zwölf kilogramm",
    "fünftes kapitel",
]

print("=== German ITN ===")
for text in examples_de:
    result = inverse_normalizer_de.inverse_normalize(text, verbose=False)
    print(f"  {text!r:35} → {result!r}")

---
## 4. Batch Processing

Use `normalize_list()` to process multiple texts efficiently with parallel jobs.

In [ ]:
texts = [
    "The package weighs 2.5 kg and costs $19.99",
    "Dr. Johnson called at 4:15pm on 03/22/2024",
    "The speed limit changes from 30 mph to 60 mph",
    "She scored 98.5% and ranked 1st in her class",
    "Add 3 tbsp of flour and 1.5 cups of milk",
]

results = normalizer.normalize_list(texts, verbose=False, n_jobs=1)

print("=== Batch Text Normalization ===")
for original, normalized in zip(texts, results):
    print(f"  IN:  {original}")
    print(f"  OUT: {normalized}")
    print()

---
## 5. TTS and ASR Pipeline Example

This shows how TN and ITN fit into real speech pipelines.

In [ ]:
# Simulated TTS pipeline: normalize text before synthesizing speech
tts_inputs = [
    "The meeting is scheduled for 14:30 on 12/05/2024",
    "The total bill is $1,250.75 including 8.5% tax",
    "Dr. Chen will present at 9am in Room 3B",
]

print("=== TTS Pre-processing (TN) ===")
print("Normalizing text before sending to a TTS model:\n")
for text in tts_inputs:
    normalized = normalizer.normalize(text, verbose=False)
    print(f"  Raw text : {text}")
    print(f"  TTS input: {normalized}")
    print()

In [ ]:
# Simulated ASR pipeline: clean up raw transcript from speech recognition
asr_outputs = [
    "the package costs twenty five dollars and ninety nine cents",
    "call me back at three forty five p m",
    "the file is two point three gigabytes",
]

print("=== ASR Post-processing (ITN) ===")
print("Cleaning up raw ASR transcript:\n")
for text in asr_outputs:
    normalized = inverse_normalizer.inverse_normalize(text, verbose=False)
    print(f"  ASR output : {text}")
    print(f"  Cleaned up : {normalized}")
    print()

---
## 6. Try It Yourself

Modify the text below and run the cell to test your own inputs.

In [ ]:
# ── Text Normalization ────────────────────────────────────────────────────────
my_text = "Change this to any text you want to normalize"
my_lang = "en"  # Options: en, de, es, fr, hu, sv, zh, ar, it, hy, ja, hi, ko, vi, pt

n = Normalizer(input_case='cased', lang=my_lang)
print(f"TN [{my_lang}]: {my_text!r}")
print(f"       → {n.normalize(my_text, verbose=False)!r}")

In [ ]:
# ── Inverse Text Normalization ────────────────────────────────────────────────
my_spoken_text = "change this to any spoken text you want to convert back"
my_itn_lang = "en"  # Options: en, de, es, fr, hu, sv, zh, ar, it, hy, ja, hi, ko, vi, pt

inv_n = InverseNormalizer(lang=my_itn_lang)
print(f"ITN [{my_itn_lang}]: {my_spoken_text!r}")
print(f"        → {inv_n.inverse_normalize(my_spoken_text, verbose=False)!r}")